# Preprocessing Data SARC (Data Penuh)

Notebook ini menyiapkan seluruh data SARC untuk training RoBERTa dan DistilBERT. Hasilnya disimpan ke `data/split_full/`.

Alur:
1. Load data mentah (1.010.826 pasangan)
2. Cleaning teks parent dan comment
3. Buang data kosong, label bertentangan, duplikat, dan artefak `/s`
4. Split 80/10/10 per kelompok comment
5. Cek kebocoran data
6. Ambil data pilot
7. Simpan hasil

Perbedaan dengan preprocessing sebelumnya (`data/split/`):

| | Sebelumnya | Sekarang |
|---|---|---|
| Cleaning teks | lowercase, hapus tanda baca, emoji, mention | hanya HTML entity, URL, tag HTML, spasi |
| Jumlah data | subset 200.000 | seluruh data |
| Cek duplikat | teks harus sama persis | beda kapital/tanda baca tetap dianggap sama |
| Split | acak per baris | per kelompok comment |
| Pilot | 40.000 | 2.000 |

## 1. Setup

| Variabel | Nilai | Keterangan |
|---|---|---|
| `SEED` | 42 | agar split selalu sama |
| `TRAIN_SIZE`, `VAL_SIZE`, `TEST_SIZE` | 0,8 / 0,1 / 0,1 | proporsi split |
| `PILOT_SIZE` | 2.000 | cukup untuk uji pipeline |
| `OUT_DIR` | `data/split_full/` | folder baru, `data/split/` tidak ditimpa |

In [1]:
import re, html, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

SEED = 42
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.8, 0.1, 0.1
PILOT_SIZE = 2_000                        # pilot kecil, hanya untuk uji pipeline (arahan dosen)

sns.set_theme(style="whitegrid")
ROOT = Path.cwd().parents[1]
RAW_PATH = ROOT / "data/raw/train-balanced-sarcasm.csv"
OUT_DIR = ROOT / "data/split_full"        # folder BARU: data/split/ dan data/processed/ (setup lama) tidak disentuh
OUT_DIR.mkdir(parents=True, exist_ok=True)
    
LABEL_NAMES = {0: "Non-Sarkas", 1: "Sarkas"}
print("Hasil akan disimpan ke:", OUT_DIR)


Hasil akan disimpan ke: /Users/hibrizi/Project/TA/data/split_full


## 2. Load Data

`raw_idx` menyimpan nomor baris asli supaya setiap data bisa dilacak kembali ke file mentah saat analisis error.

In [2]:
df = pd.read_csv(RAW_PATH)

df["raw_idx"] = df.index          # ID baris asli, untuk penelusuran saat analisis error
print(df.shape)
df[["label", "parent_comment", "comment"]].head()

(1010826, 11)


,label,parent_comment,comment
0,0,"Yeah, I get that argument. At this point, I'd ...",NC and NH.
1,0,The blazers and Mavericks (The wests 5 and 6 s...,You do know west teams play against west teams...
2,0,They're favored to win.,"They were underdogs earlier today, but since G..."
3,0,deadass don't kill my buzz,"This meme isn't funny none of the ""new york ni..."
4,0,Yep can confirm I saw the tool they use for th...,I could use one of those tools.


#### Insight
- Data mentah berisi 1.010.826 baris dan 11 kolom (10 kolom asli + `raw_idx`).
- Yang dipakai untuk model hanya `parent_comment`, `comment`, dan `label` (1 = sarkas).

## 3. Fungsi Text Cleaning

Langkah cleaning (sama untuk parent dan comment):

| No | Langkah | Contoh |
|---|---|---|
| 1 | HTML entity → karakter asli (2 kali, karena ada yang tersimpan dua lapis) | `&amp;gt;` → `>` |
| 2 | Link markdown → teksnya saja | `[link](http://x.com)` → `link` |
| 3 | Hapus URL | `lihat https://...` → `lihat` |
| 4 | Hapus tag HTML | `<b>bold</b>` → `bold` |
| 5 | Rapikan spasi | `oh   great` → `oh great` |

Huruf kapital, tanda baca, dan mention tidak diubah karena bisa menjadi petunjuk sarkasme (misalnya `"OH GREAT..."`). Stopword, stemming, dan lemmatization juga tidak dilakukan.

Beberapa catatan:
- Pola URL hanya cocok jika `http` atau `www.` ada di awal kata, supaya kata seperti `"Awww..."` tidak ikut terhapus.
- Tag HTML yang dihapus hanya tag asli (`b`, `i`, `br`, `a`, dll.), sehingga `"<3"` dan `"</sarcasm>"` tetap ada.
- `loose_key` membuat versi teks tanpa kapital dan tanda baca. Fungsi ini hanya dipakai untuk mencari duplikat dan artefak, bukan sebagai input model.
- `is_artifact` mendeteksi artefak `/s`. Di Reddit, balasan seperti *"You forgot the /s"* kehilangan "/s"-nya saat dataset dibuat, sehingga tersisa *"You forgot the"* dengan label sarkas. Aturannya: kalimat maksimal 8 kata dengan *forgot/dropped/missed* yang tidak diikuti benda.

In [3]:
URL_RE = r"(?i)\b(?:https?://|www\.)\S+"                              # \b: "Awww..." tidak dianggap URL
MD_LINK_RE = r"\[([^\]]*)\]\(\s*(?:https?://|www\.)[^)\s]*\s*\)"      # [teks](url) -> teks
MD_LINK_ANY = r"\[[^\]]*\]\(\s*(?:https?://|www\.)"                   # versi tanpa grup, hanya untuk menghitung
HTML_TAGS = "a|abbr|b|blockquote|br|code|del|div|em|font|h[1-6]|hr|i|img|li|ol|p|pre|s|span|strike|strong|sub|sup|table|td|th|tr|u|ul"
HTML_TAG_RE = rf"(?i)</?(?:{HTML_TAGS})\b[^<>]*>"                     # hanya tag HTML asli; "<3" dan "</sarcasm>" aman
ENTITY_RE = r"&(?:[a-zA-Z]+|#\d+);"                                   # kode HTML, misalnya &gt; &amp; &#39;

# Artefak "/s": balasan pendek (<= 8 kata) yang kata kerjanya "menggantung" tanpa benda,
# karena bendanya ("/s") sudah dihapus pembuat dataset. Contoh: "You forgot the", "Here, you dropped this".
DET = r"(?:the|your|this|these|a|an|my|his|her|their|ur|you)"
ARTIFACT_RE = (rf"(?:\w+ ){{0,5}}(?:forgot|dropped|missed)(?: out)?"
               rf"(?: to(?: (?:add|put|include|use|write|tag|append))?(?: {DET})?| {DET})"
               rf"|(?:\w+ ){{0,3}}(?:you|u|he|she|they) (?:forgot|dropped|missed)(?: (?:this|it))?")
ARTIFACT_MAX_WORDS = 8

# Hanya untuk INFO (dihitung, tidak dihapus)
MENTION_RE = r"(?:^|\s)(?:/?u/\w+|@\w+)"

def unescape_twice(t: str) -> str:
    return html.unescape(html.unescape(t))                   # sebagian teks tersimpan 2 lapis: &amp;gt; -> &gt; -> >

def clean_text(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str).map(unescape_twice)          # 1. kode HTML -> karakter asli
    s = s.str.replace(MD_LINK_RE, r"\1", regex=True)          # 2a. link markdown -> teksnya saja
    s = s.str.replace(URL_RE, " ", regex=True)                # 2b. hapus URL
    s = s.str.replace(HTML_TAG_RE, " ", regex=True)           # 3. hapus tag HTML
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()    # 4. rapikan spasi
    return s                                                  # huruf kapital, tanda baca, mention TETAP

def loose_key(s: pd.Series) -> pd.Series:
    """Versi 'kasar' (huruf kecil, tanpa tanda baca) HANYA untuk mencari data kembar & artefak. Bukan input model."""
    k = (s.str.lower()
          .str.replace(r"[^\w\s]", " ", regex=True)
          .str.replace(r"\s+", " ", regex=True).str.strip())
    return k.where(k != "", s)                                # teks yang isinya hanya tanda baca (misal "?") tetap dibedakan

def is_artifact(s: pd.Series) -> pd.Series:
    k = loose_key(s)
    return k.str.fullmatch(ARTIFACT_RE) & (k.str.split().str.len() <= ARTIFACT_MAX_WORDS)

# Uji cepat fungsi
print(clean_text(pd.Series([
    "Oh GREAT, another &gt; quote... https://reddit.com/r/x",
    "Thanks @someone and /u/other_user!!! Awww...",
    "Check [this link](http://example.com/a) &amp;amp; <b>bold</b>   text",
    "I <3 you <i>so</i> much </sarcasm>",
])).tolist())
print(is_artifact(pd.Series([
    "You forgot the", "Here, you dropped this.", "I think you forgot to add",
    "Never forget.", "I forgot that.", "You forgot the milk",
])).tolist())


['Oh GREAT, another > quote...', 'Thanks @someone and /u/other_user!!! Awww...', 'Check this link & bold text', 'I <3 you so much </sarcasm>']
[True, True, True, False, False, False]


#### Insight
- `"Oh GREAT, another &gt; quote... https://reddit.com/r/x"` menjadi `"Oh GREAT, another > quote..."`. Kapital dan `...` tetap ada.
- `"Thanks @someone and /u/other_user!!! Awww..."` tidak berubah.
- `"Check [this link](http://example.com/a) &amp;amp; <b>bold</b>   text"` menjadi `"Check this link & bold text"`.
- `"I <3 you <i>so</i> much </sarcasm>"` menjadi `"I <3 you so much </sarcasm>"`.
- `is_artifact` menandai *"You forgot the"*, *"Here, you dropped this."*, dan *"I think you forgot to add"*, tetapi tidak menandai *"Never forget."*, *"I forgot that."*, dan *"You forgot the milk"*.

## 4. Jumlah Teks yang Terkena Cleaning

Menghitung berapa teks di data mentah yang mengandung HTML entity, URL, tag HTML, dan sebagainya.

In [4]:
impact = {}
for col in ["comment", "parent_comment"]:
    raw = df[col].fillna("").astype(str)
    unesc = raw.map(unescape_twice)
    impact[col] = {
        "Ada kode HTML (&gt;, &amp;, ...)": int((raw != unesc).sum()),
        "Ada kode HTML 2 lapis (&amp;gt;)": int(raw.str.contains(r"&amp;(?:[a-zA-Z]+|#\d+);").sum()),
        "Ada link markdown [teks](url)": int(unesc.str.contains(MD_LINK_ANY).sum()),
        "Ada URL": int(unesc.str.contains(URL_RE).sum()),
        "Ada tag HTML": int(unesc.str.contains(HTML_TAG_RE).sum()),
        "Ada spasi berlebih": int(unesc.str.contains(r"\s{2,}|^\s|\s$").sum()),
        "Ada emoji / karakter non-ASCII": int(raw.str.contains(r"[^\x00-\x7F]").sum()),
    }
impact = pd.DataFrame(impact)
impact["% dari data mentah (comment)"] = (impact["comment"] / len(df) * 100).round(3)
display(impact)


,comment,parent_comment,% dari data mentah (comment)
"Ada kode HTML (&gt;, &amp;, ...)",2859,7209,0.283
Ada kode HTML 2 lapis (&amp;gt;),171,264,0.017
Ada link markdown [teks](url),88,102,0.009
Ada URL,99,136,0.010
Ada tag HTML,0,0,0.000
Ada spasi berlebih,36,133,0.004
Ada emoji / karakter non-ASCII,0,0,0.000


#### Insight
- HTML entity ada di 2.859 comment (0,28%) dan 7.209 parent. Yang dua lapis hanya 171 comment dan 264 parent.
- URL ada di 99 comment dan 136 parent, link markdown di 88 comment dan 102 parent.
- Tidak ada tag HTML. Spasi berlebih hanya 36 comment dan 133 parent.
- Tidak ada emoji maupun karakter non-ASCII sama sekali, jadi dataset ini memang tidak mengandung emoji.
- Secara keseluruhan cleaning hanya mengubah sebagian kecil teks.

## 5. Cleaning Data

Data bermasalah dibuang satu per satu dan dicatat di log. Pengecekan label bertentangan dan duplikat memakai `comment_key` dan `parent_key`, jadi `"Yes."` dan `"yes"` pada parent yang sama dihitung sebagai pasangan yang sama.

In [5]:
log = []
def record(step, data):
    log.append({"Langkah": step, "Sisa baris": len(data)})

df = df.copy()
record("Data mentah", df)

df = df.dropna(subset=["comment"])
record("Hapus comment kosong (NaN)", df)

df["comment_clean"] = clean_text(df["comment"])
df["parent_clean"] = clean_text(df["parent_comment"])
df = df[(df["comment_clean"] != "") & (df["parent_clean"] != "")]
record("Hapus teks kosong setelah cleaning", df)

df["comment_key"] = loose_key(df["comment_clean"])        # kunci pengecekan kembar (bukan input model)
df["parent_key"] = loose_key(df["parent_clean"])
PAIR_KEY = ["comment_key", "parent_key"]

conflict = df.groupby(PAIR_KEY)["label"].transform("nunique") > 1
df = df[~conflict]
record("Hapus pasangan (kembar) berlabel bertentangan", df)

df = df.drop_duplicates(PAIR_KEY)
record("Hapus pasangan duplikat / kembar", df)

df = df[~is_artifact(df["comment_clean"])]
record("Hapus artefak /s", df)

df = df.reset_index(drop=True)

preprocessing_log = pd.DataFrame(log)
preprocessing_log["Dibuang"] = (-preprocessing_log["Sisa baris"].diff()).fillna(0).astype(int)
display(preprocessing_log.style.hide(axis="index"))


Langkah,Sisa baris,Dibuang
Data mentah,1010826,0
Hapus comment kosong (NaN),1010771,55
Hapus teks kosong setelah cleaning,1010762,9
Hapus pasangan (kembar) berlabel bertentangan,1010620,142
Hapus pasangan duplikat / kembar,1009463,1157
Hapus artefak /s,1000864,8599


#### Insight

| Langkah | Dibuang | Sebelumnya |
|---|---:|---:|
| Comment kosong (NaN) | 55 | 55 |
| Teks kosong setelah cleaning | 9 | 1.409 |
| Label bertentangan | 142 | 142 |
| Duplikat | 1.157 | 1.147 |
| Artefak `/s` | 8.599 | 3.825 |

- Total dibuang 9.962 baris (0,99%), tersisa 1.000.864 pasangan.
- Teks kosong turun jauh karena tanda baca tidak lagi dihapus, jadi teks seperti `"?"` atau `":)"` tetap ada.
- Artefak `/s` yang terdeteksi naik lebih dari dua kali lipat karena aturannya sekarang mencakup variasi seperti *"you forgot"*, *"forgot the"*, dan *"here you dropped this"*.

## 6. Sanity Check

Bagian A harus bernilai 0 (kalau tidak, `assert` menghentikan notebook). Bagian B memastikan kapital, tanda baca, dan mention masih ada setelah cleaning.

In [6]:
def count_both(pattern):
    return int(df["comment_clean"].str.contains(pattern, regex=True).sum()
               + df["parent_clean"].str.contains(pattern, regex=True).sum())

# A. Yang HARUS sudah hilang
checks = pd.Series({
    "Masih ada URL": count_both(URL_RE),       # "http://" yang disebut dalam kalimat (tanpa alamat) bukan URL
    "Masih ada tag HTML": count_both(HTML_TAG_RE),
    "Masih ada spasi ganda / di ujung": count_both(r"\s{2,}|^\s|\s$"),
    "Masih ada teks kosong": int((df["comment_clean"] == "").sum() + (df["parent_clean"] == "").sum()),
    "Masih ada pasangan kembar": int(df.duplicated(PAIR_KEY).sum()),
    "Masih ada artefak /s": int(is_artifact(df["comment_clean"]).sum()),
}, name="Jumlah (harus 0)")
display(checks.to_frame())
assert checks.sum() == 0, "Masih ada data yang belum bersih!"

# B. Yang HARUS tetap ada (bukti tidak ikut terhapus)
kept = pd.Series({
    "Huruf kapital": count_both(r"[A-Z]"),
    "Tanda baca": count_both(r"[^\w\s]"),
    "Kata KAPITAL semua (>= 3 huruf)": count_both(r"\b[A-Z]{3,}\b"),
    "Tanda baca berulang (!!, ??, ...)": count_both(r"[!?]{2,}|\.{3,}"),
    "Mention (u/... atau @...)": count_both(MENTION_RE),
    "Sisa kode HTML (info)": count_both(ENTITY_RE),
}, name="Jumlah teks (comment + parent)")
display(kept.to_frame())

print("Proporsi label setelah cleaning:")
display(df["label"].map(LABEL_NAMES).value_counts().to_frame("Jumlah")
          .assign(Persen=lambda t: (t["Jumlah"] / len(df) * 100).round(2)))

changed = df[df["comment"] != df["comment_clean"]]
print(f"Comment yang berubah karena cleaning: {len(changed):,} dari {len(df):,}")
with pd.option_context("display.max_colwidth", 100):
    display(changed[["comment", "comment_clean"]].sample(min(8, len(changed)), random_state=SEED))


,Jumlah (harus 0)
Masih ada URL,0
Masih ada tag HTML,0
Masih ada spasi ganda / di ujung,0
Masih ada teks kosong,0
Masih ada pasangan kembar,0
Masih ada artefak /s,0


,Jumlah teks (comment + parent)
Huruf kapital,1863567
Tanda baca,1731703
Kata KAPITAL semua (>= 3 huruf),220088
"Tanda baca berulang (!!, ??, ...)",156369
Mention (u/... atau @...),1492
Sisa kode HTML (info),1


Proporsi label setelah cleaning:


,Jumlah,Persen
label,,
Non-Sarkas,504502,50.41
Sarkas,496362,49.59


Comment yang berubah karena cleaning: 2,942 dari 1,000,864


,comment,comment_clean
807247,playing shaco for the first time in like 2 years :&lt; probably bad tho,playing shaco for the first time in like 2 years :< probably bad tho
228136,&lt;3,<3
685903,Her evidence from a rando person trumps your lived reality &amp; facts!,Her evidence from a rando person trumps your lived reality & facts!
61232,Theres a reason the game is called Investment Bankers &amp;&amp; Waifus,Theres a reason the game is called Investment Bankers && Waifus
508349,With one the abortionist is mother &amp; Planned Parenthood (bad) in the other situation the abo...,With one the abortionist is mother & Planned Parenthood (bad) in the other situation the abortio...
188380,"Got at&amp;t which offers NBCSN and i watch every practice, quali and race of the season.","Got at&t which offers NBCSN and i watch every practice, quali and race of the season."
829683,Deagle or 500 S&amp;W revolver option plz!,Deagle or 500 S&W revolver option plz!
582240,"Forget serial killer, i'm worried about thieves now bypassing the breaking part of breaking &amp...","Forget serial killer, i'm worried about thieves now bypassing the breaking part of breaking & en..."


#### Insight
- Semua pemeriksaan bagian A bernilai 0.
- Dari 2.001.728 teks (comment + parent): 1.863.567 mengandung huruf kapital, 1.731.703 mengandung tanda baca, 220.088 punya kata kapital semua, 156.369 punya `!!`, `??`, atau `...`, dan 1.492 punya mention.
- Tersisa 1 teks dengan HTML entity karena tersimpan tiga lapis (`"Army &amp; Navy"`). Jumlahnya bisa diabaikan.
- Proporsi label: 50,41% non-sarkas dan 49,59% sarkas. Sarkas sedikit berkurang karena artefak `/s` yang dibuang hampir semuanya berlabel sarkas (8.580 dari 8.599).
- Hanya 2.942 comment (0,29%) yang berubah, misalnya `"&lt;3"` → `"<3"` dan `"S&amp;W"` → `"S&W"`.

## 7. Split 80/10/10 per Kelompok Comment

Banyak comment pendek muncul berulang dengan parent berbeda, misalnya `"Yes."`, `"Thanks!"`, dan `"lol"`. Kalau split dilakukan per baris, comment yang sama bisa ada di train dan test sekaligus, dan untuk input comment saja model bisa menebak dari hafalan. Karena itu split dilakukan per kelompok comment (berdasarkan `comment_key`), sehingga comment yang sama selalu berada di split yang sama.

Supaya proporsi label tetap seimbang, kelompok di-stratify: kelompok 1 baris berdasarkan labelnya, kelompok lebih dari 1 baris berdasarkan label mayoritasnya.

In [7]:
# Split per KELOMPOK comment: comment yang sama/kembar (misal "Yes." dan "yes") selalu masuk split yang sama.
groups = (df.groupby("comment_key")["label"]
            .agg(n="size", pct_sarkas="mean")
            .reset_index())
# Strata agar proporsi sarkas tetap seimbang:
# kelompok berisi 1 baris -> labelnya; kelompok berisi banyak baris -> mayoritas sarkas atau bukan
groups["strata"] = np.where(groups["n"] == 1,
                            "tunggal_" + groups["pct_sarkas"].round().astype(int).astype(str),
                            "banyak_" + (groups["pct_sarkas"] >= 0.5).astype(int).astype(str))

g_train, g_temp = train_test_split(
    groups, train_size=TRAIN_SIZE, stratify=groups["strata"], random_state=SEED
)
g_val, g_test = train_test_split(
    g_temp, test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE), stratify=g_temp["strata"], random_state=SEED
)
split_of = pd.concat([
    pd.Series("train", index=g_train["comment_key"]),
    pd.Series("val", index=g_val["comment_key"]),
    pd.Series("test", index=g_test["comment_key"]),
])
df["split"] = df["comment_key"].map(split_of)
assert df["split"].notna().all()

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]
splits = {"train": train_df, "val": val_df, "test": test_df}

split_summary = pd.DataFrame({
    name: {
        "Jumlah": len(d),
        "% dari total": len(d) / len(df) * 100,
        "% Sarkas": d["label"].mean() * 100,
        "Median kata comment": d["comment_clean"].str.split().str.len().median(),
        "Median kata parent": d["parent_clean"].str.split().str.len().median(),
    }
    for name, d in splits.items()
}).T.round(2)
display(split_summary)


,Jumlah,% dari total,% Sarkas,Median kata comment,Median kata parent
train,799459.0,79.88,49.66,9.0,14.0
val,100276.0,10.02,49.38,9.0,14.0
test,101129.0,10.10,49.27,9.0,14.0


#### Insight
- Train 799.459 (79,88%), val 100.276 (10,02%), test 101.129 (10,10%).
- Proporsi sarkas 49,66% / 49,38% / 49,27%. Selisih sekitar 0,4 poin terjadi karena kelompok besar seperti `"Yes."` harus masuk utuh ke satu split.
- Median panjang comment (9 kata) dan parent (14 kata) sama di ketiga split.

## 8. Cek Kebocoran Data

Empat pemeriksaan yang harus 0:
- pasangan parent–comment yang sama persis,
- pasangan yang hanya beda kapital/tanda baca,
- `raw_idx` yang sama,
- comment yang sama (atau hanya beda kapital/tanda baca).

Parent yang sama di dua split hanya dicatat sebagai informasi, karena comment dan labelnya berbeda.

In [8]:
def overlap(col_fn):
    keys = {name: col_fn(d) for name, d in splits.items()}
    return {
        "train–val": len(keys["train"] & keys["val"]),
        "train–test": len(keys["train"] & keys["test"]),
        "val–test": len(keys["val"] & keys["test"]),
    }

leak = pd.DataFrame({
    "Pasangan sama": overlap(lambda d: set(zip(d["comment_clean"], d["parent_clean"]))),
    "Pasangan kembar": overlap(lambda d: set(zip(d["comment_key"], d["parent_key"]))),
    "raw_idx sama": overlap(lambda d: set(d["raw_idx"])),
    "Comment sama/kembar": overlap(lambda d: set(d["comment_key"])),
})
leak_info = pd.DataFrame({"Parent sama (info, bukan bocoran)": overlap(lambda d: set(d["parent_clean"]))})
display(leak.add_suffix(" (harus 0)").join(leak_info))
assert leak.to_numpy().sum() == 0, "Ada kebocoran data antar split!"


,Pasangan sama (harus 0),Pasangan kembar (harus 0),raw_idx sama (harus 0),Comment sama/kembar (harus 0),"Parent sama (info, bukan bocoran)"
train–val,0,0,0,0,3108
train–test,0,0,0,0,3094
val–test,0,0,0,0,643


#### Insight
- Keempat pemeriksaan bernilai 0 di semua pasangan split.
- Kalau data dibagi acak per baris, comment yang sama (misalnya "Yes.") bisa masuk ke train dan test sekaligus. Split per kelompok mencegah hal itu, terlihat dari kolom comment sama/kembar yang bernilai 0.
- Parent yang sama muncul di train–val (3.108), train–test (3.094), dan val–test (643). Ini wajar karena satu postingan bisa dibalas banyak orang.

## 9. Data Pilot

Pilot diambil dari train dan hanya dipakai untuk mengecek pipeline (tokenisasi, training, evaluasi) sebelum training penuh.

In [9]:
pilot_df, _ = train_test_split(
    train_df, train_size=PILOT_SIZE, stratify=train_df["label"], random_state=SEED
)
print("Pilot:", pilot_df.shape, "| % sarkas:", round(pilot_df["label"].mean() * 100, 2))


Pilot: (2000, 16) | % sarkas: 49.65


#### Insight
- Pilot berisi 2.000 pasangan dengan 49,65% sarkas, mirip dengan train.

## 10. Simpan Hasil

Kolom yang disimpan:

| Kolom | Keterangan |
|---|---|
| `parent_clean`, `comment_clean` | input model |
| `label` | 1 = sarkas, 0 = non-sarkas |
| `raw_idx` | nomor baris asli |
| `parent_comment`, `comment` | teks asli, untuk analisis error |
| `subreddit`, `date` | informasi tambahan, tidak dipakai saat training |

File pendukung: `preprocessing_log.csv`, `cleaning_impact.csv`, `split_summary.csv`, dan `split_config.json`.

In [10]:
KEEP_COLS = ["raw_idx", "label", "parent_clean", "comment_clean",
             "parent_comment", "comment", "subreddit", "date"]
all_splits = {**splits, "train_pilot": pilot_df}

for name, d in all_splits.items():
    path = OUT_DIR / f"{name}.parquet"
    d[KEEP_COLS].reset_index(drop=True).to_parquet(path, index=False)
    print(f"{name:12s} {len(d):>9,} baris -> {path}")

preprocessing_log.to_csv(OUT_DIR / "preprocessing_log.csv", index=False)
split_summary.to_csv(OUT_DIR / "split_summary.csv")
impact.to_csv(OUT_DIR / "cleaning_impact.csv")

config = {
    "seed": SEED,
    "subset": None,                                   # seluruh data, tanpa subset (arahan dosen)
    "split_ratio": {"train": TRAIN_SIZE, "val": VAL_SIZE, "test": TEST_SIZE},
    "split_method": "stratified per kelompok comment (comment sama/kembar selalu di split yang sama)",
    "pilot_size": PILOT_SIZE,
    "n_raw": int(preprocessing_log["Sisa baris"].iloc[0]),   # baris pertama log = jumlah data mentah
    "n_after_cleaning": len(df),
    "n_rows": {name: len(d) for name, d in all_splits.items()},
    "text_columns": {"parent": "parent_clean", "comment": "comment_clean"},
    "preprocessing": ["html_unescape_x2", "markdown_link_to_text", "remove_url", "remove_html_tag",
                      "normalize_whitespace", "drop_empty", "drop_conflicting_label_loose",
                      "drop_duplicate_pair_loose", "drop_s_artifact_v2"],
    "kept": ["case", "punctuation", "mention", "emoji (tidak ada di data)"],
    "duplicate_key": "huruf kecil + tanpa tanda baca (hanya untuk pengecekan; input model tetap teks asli)",
    "artifact_regex": ARTIFACT_RE,
    "artifact_max_words": ARTIFACT_MAX_WORDS,
}
(OUT_DIR / "split_config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))


train          799,459 baris -> /Users/hibrizi/Project/TA/data/split_full/train.parquet
val            100,276 baris -> /Users/hibrizi/Project/TA/data/split_full/val.parquet
test           101,129 baris -> /Users/hibrizi/Project/TA/data/split_full/test.parquet
train_pilot      2,000 baris -> /Users/hibrizi/Project/TA/data/split_full/train_pilot.parquet
{
  "seed": 42,
  "subset": null,
  "split_ratio": {
    "train": 0.8,
    "val": 0.1,
    "test": 0.1
  },
  "split_method": "stratified per kelompok comment (comment sama/kembar selalu di split yang sama)",
  "pilot_size": 2000,
  "n_raw": 1010826,
  "n_after_cleaning": 1000864,
  "n_rows": {
    "train": 799459,
    "val": 100276,
    "test": 101129,
    "train_pilot": 2000
  },
  "text_columns": {
    "parent": "parent_clean",
    "comment": "comment_clean"
  },
  "preprocessing": [
    "html_unescape_x2",
    "markdown_link_to_text",
    "remove_url",
    "remove_html_tag",
    "normalize_whitespace",
    "drop_empty",
    "drop_con

#### Insight
- Tersimpan train 799.459, val 100.276, test 101.129, dan train_pilot 2.000 baris.
- `split_config.json` mencatat 1.010.826 data mentah dan 1.000.864 data bersih, beserta langkah cleaning dan regex artefak `/s`.

## 11. Verifikasi File

In [11]:
for name in all_splits:
    check = pd.read_parquet(OUT_DIR / f"{name}.parquet")
    print(f"{name:12s} {check.shape} | % sarkas {check['label'].mean() * 100:.2f}")
check.head()


train        (799459, 8) | % sarkas 49.66
val          (100276, 8) | % sarkas 49.38
test         (101129, 8) | % sarkas 49.27
train_pilot  (2000, 8) | % sarkas 49.65


,raw_idx,label,parent_clean,comment_clean,parent_comment,comment,subreddit,date
0,419126,1,"My husband of one year, together 5 years, I do...",Definitely a CIA agent.,"My husband of one year, together 5 years, I do...",Definitely a CIA agent.,relationships,2016-02
1,787626,0,What do you guys think of this? Cars that repo...,Someone has to make sure the police follow dri...,What do you guys think of this? Cars that repo...,Someone has to make sure the police follow dri...,ProtectAndServe,2014-11
2,402819,0,UPDATE: Based black supporter at Chicago Trump...,"**2016, THE YEAR /POL/ MEMES A MAN INTO OFFICE**",UPDATE: Based black supporter at Chicago Trump...,"**2016, THE YEAR /POL/ MEMES A MAN INTO OFFICE**",The_Donald,2016-03
3,775038,0,"I'm pretty sure it's called the inbox, ""homes.""",Click it and find out!,"I'm pretty sure it's called the inbox, ""homes.""",Click it and find out!,GrandTheftAutoV,2014-11
4,865824,0,Recently visited my brother's place. This is w...,"Might be an issue with the line, I had somethi...",Recently visited my brother's place. This is w...,"Might be an issue with the line, I had somethi...",Comcast,2014-04


#### Insight
- Semua file terbaca kembali dengan 8 kolom dan jumlah baris yang sesuai.

## 12. Kesimpulan

1. Cleaning hanya menyentuh HTML entity, URL, tag HTML, dan spasi. Kapital, tanda baca, dan mention tetap ada. Dataset tidak mengandung emoji.
2. Dibuang 9.962 baris (0,99%), sebagian besar artefak `/s` (8.599).
3. Data bersih 1.000.864 pasangan dibagi menjadi 799.459 / 100.276 / 101.129 dengan proporsi sarkas 49,3–49,7%.
4. Tidak ada kebocoran data antar split, termasuk comment yang sama.

Langkah berikutnya adalah tokenisasi untuk tiga jenis input (comment saja, early fusion, late fusion) dengan `roberta-base` dan `distilbert-base-cased`, sekaligus mengecek apakah `max_length` 256 masih cukup.